# Step 4: Analyze Results & Generate Plots

**CMPE 260 — Group 6**

This notebook reads the CSVs from Steps 1-3 and generates:
- Training curve comparison plots
- Degradation curves (shift level vs score)
- Robustness heatmap
- AUDC comparison bar chart
- Summary tables for the report

**No GPU needed** — runs on CPU or locally.

**Prerequisites:** Upload CSVs from Steps 1-3 to `results/` folder.

In [ ]:
import csv, glob, os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

ENVIRONMENTS = ['hopper-medium-v2', 'halfcheetah-medium-v2', 'walker2d-medium-v2']
GRAVITY_LEVELS = [0.5, 1.0, 1.5, 2.0]
OBS_NOISE_LEVELS = [0.0, 0.01, 0.1, 0.3]
SEEDS = [42, 43, 44]

def load_csv(filepath):
    rows = []
    with open(filepath) as f:
        for row in csv.DictReader(f):
            for k in row:
                try: row[k] = float(row[k])
                except: pass
            rows.append(row)
    return rows

def compute_audc(results, shift_type):
    filtered = sorted([r for r in results if r['shift_type'] == shift_type],
                      key=lambda x: x['shift_level'])
    if len(filtered) < 2: return 0.0
    return np.trapz([abs(r['robustness_drop']) for r in filtered],
                    [r['shift_level'] for r in filtered])

print('Ready. Upload CSVs to results/ folder.')


In [ ]:
# Upload CSVs if running on Colab
# from google.colab import files
# uploaded = files.upload()  # Upload all CSVs from Steps 1-3
# os.makedirs('results', exist_ok=True)
# for name, data in uploaded.items():
#     with open(f'results/{name}', 'wb') as f: f.write(data)

# List available results
print('Available result files:')
for f in sorted(glob.glob('results/*.csv')):
    print(f'  {f}')

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, env_name in enumerate(ENVIRONMENTS):
    ax = axes[idx]
    for nq, color, style in [(2, 'steelblue', '-o'), (3, 'coral', '--s')]:
        all_runs = []
        for seed in SEEDS:
            fpath = f'results/training_{env_name}_{nq}Q_s{seed}.csv'
            if os.path.exists(fpath):
                data = load_csv(fpath)
                all_runs.append([r['normalized_score'] for r in data])
                steps = [r['step'] for r in data]
        if all_runs:
            all_runs = np.array(all_runs)
            mean_scores = all_runs.mean(axis=0)
            std_scores = all_runs.std(axis=0)
            ax.plot(steps, mean_scores, style, label=f'{nq}Q', color=color, linewidth=2, markersize=6)
            ax.fill_between(steps, mean_scores - std_scores, mean_scores + std_scores, color=color, alpha=0.2)
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Normalized Score')
    ax.set_title(env_name)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training Curves — Baseline (2Q) vs Q-Ensemble (3Q)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plot_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## Degradation Curves

In [ ]:
fig, axes = plt.subplots(len(ENVIRONMENTS), 2, figsize=(14, 5*len(ENVIRONMENTS)))

for row, env_name in enumerate(ENVIRONMENTS):
    for col, (shift_type, levels) in enumerate([('gravity', GRAVITY_LEVELS), ('obs_noise', OBS_NOISE_LEVELS)]):
        ax = axes[row, col] if len(ENVIRONMENTS) > 1 else axes[col]
        for nq, color, style in [(2, 'steelblue', '-o'), (3, 'coral', '--s')]:
            level_returns = {l: [] for l in levels}
            for seed in SEEDS:
                fpath = f'results/shift_{env_name}_{nq}Q_s{seed}.csv'
                if os.path.exists(fpath):
                    data = load_csv(fpath)
                    filtered = [r for r in data if r['shift_type'] == shift_type]
                    for r in filtered:
                        level_returns[r['shift_level']].append(r['raw_return'])
            
            means = []
            stds = []
            valid_levels = []
            for l in levels:
                if level_returns[l]:
                    means.append(np.mean(level_returns[l]))
                    stds.append(np.std(level_returns[l]))
                    valid_levels.append(l)
            if valid_levels:
                ax.errorbar(valid_levels, means, yerr=stds, fmt=style, label=f'{nq}Q', color=color, linewidth=2, markersize=8, capsize=5)
        ax.set_xlabel('Gravity Scale' if shift_type == 'gravity' else 'Noise σ')
        ax.set_ylabel('Normalized Score')
        ax.set_title(f'{env_name} — {shift_type}')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle('Performance Under Distribution Shift (Mean ± Std)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/plot_degradation_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## Robustness Heatmap

In [ ]:
# Build heatmap: rows = (env, critic), cols = shift conditions
shift_labels = [f'G={l}' for l in GRAVITY_LEVELS] + [f'N={l}' for l in OBS_NOISE_LEVELS]
row_labels = []
heatmap_data = []

for env_name in ENVIRONMENTS:
    short = env_name.split('-')[0].capitalize()
    for nq in [2, 3]:
        row = []
        has_data = False
        for st, levels in [('gravity', GRAVITY_LEVELS), ('obs_noise', OBS_NOISE_LEVELS)]:
            for level in levels:
                drops = []
                for seed in SEEDS:
                    fpath = f'results/shift_{env_name}_{nq}Q_s{seed}.csv'
                    if os.path.exists(fpath):
                        has_data = True
                        data = load_csv(fpath)
                        match = [r for r in data if r['shift_type'] == st and r['shift_level'] == level]
                        if match:
                            drops.append(match[0]['robustness_drop'])
                row.append(np.mean(drops) if drops else float('nan'))
        if has_data:
            heatmap_data.append(row)
            row_labels.append(f'{short} {nq}Q')

if heatmap_data:
    fig, ax = plt.subplots(figsize=(14, max(4, len(row_labels) * 0.6)))
    data_arr = np.array(heatmap_data)
    im = ax.imshow(data_arr, cmap='RdYlGn_r', aspect='auto', vmin=-0.5, vmax=1.0)
    ax.set_xticks(range(len(shift_labels)))
    ax.set_xticklabels(shift_labels, rotation=45, ha='right')
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels)
    for i in range(len(row_labels)):
        for j in range(len(shift_labels)):
            val = data_arr[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        color='white' if abs(val) > 0.3 else 'black', fontsize=9)
    plt.colorbar(im, ax=ax, label='Δ(δ) Robustness Drop')
    ax.set_title('Robustness Drop Heatmap — All Environments & Shift Conditions')
    plt.tight_layout()
    plt.savefig('results/plot_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No shift results found. Run Step 3 first.')


## AUDC Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for col, shift_type in enumerate(['gravity', 'obs_noise']):
    ax = axes[col]
    env_labels = []
    audc_2q_mean, audc_2q_err = [], []
    audc_3q_mean, audc_3q_err = [], []
    
    for env_name in ENVIRONMENTS:
        short = env_name.split('-')[0].capitalize()
        env_labels.append(short)
        for nq, mean_lst, err_lst in [(2, audc_2q_mean, audc_2q_err), (3, audc_3q_mean, audc_3q_err)]:
            audc_vals = []
            for seed in SEEDS:
                fpath = f'results/shift_{env_name}_{nq}Q_s{seed}.csv'
                if os.path.exists(fpath):
                    audc_vals.append(compute_audc(load_csv(fpath), shift_type))
            if audc_vals:
                mean_lst.append(np.mean(audc_vals))
                err_lst.append(np.std(audc_vals))
            else:
                mean_lst.append(0)
                err_lst.append(0)

    x = np.arange(len(env_labels))
    w = 0.35
    ax.bar(x - w/2, audc_2q_mean, w, yerr=audc_2q_err, capsize=5, label='Baseline (2Q)', color='steelblue')
    ax.bar(x + w/2, audc_3q_mean, w, yerr=audc_3q_err, capsize=5, label='Q-Ensemble (3Q)', color='coral')
    ax.set_xticks(x)
    ax.set_xticklabels(env_labels)
    ax.set_ylabel('AUDC (lower = more robust)')
    ax.set_title(f'{shift_type.replace("_", " ").title()} — AUDC')
    ax.legend()

plt.suptitle('Area Under Degradation Curve', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plot_audc.png', dpi=150, bbox_inches='tight')
plt.show()


## Summary Table (for the report)

In [ ]:
print(f"\n{'='*90}")
print(f"FULL RESULTS TABLE (Mean ± Std over {len(SEEDS)} seeds)")
print(f"{'='*90}")
print(f"{'Environment':<25} {'Shift':<12} {'Level':<8} {'2Q Score':<15} {'2Q Δ(δ)':<10} {'3Q Score':<15} {'3Q Δ(δ)':<10}")
print(f"{'-'*89}")

for env_name in ENVIRONMENTS:
    for st, levels in [('gravity', GRAVITY_LEVELS), ('obs_noise', OBS_NOISE_LEVELS)]:
        for level in levels:
            scores_2q = []
            drops_2q = []
            scores_3q = []
            drops_3q = []
            for seed in SEEDS:
                f_2q = f'results/shift_{env_name}_2Q_s{seed}.csv'
                f_3q = f'results/shift_{env_name}_3Q_s{seed}.csv'
                if os.path.exists(f_2q):
                    data2 = load_csv(f_2q)
                    r = next((row for row in data2 if row['shift_type'] == st and row['shift_level'] == level), None)
                    if r: 
                        scores_2q.append(r['raw_return'])
                        drops_2q.append(r['robustness_drop'])
                if os.path.exists(f_3q):
                    data3 = load_csv(f_3q)
                    r = next((row for row in data3 if row['shift_type'] == st and row['shift_level'] == level), None)
                    if r:
                        scores_3q.append(r['raw_return'])
                        drops_3q.append(r['robustness_drop'])
                        
            s2 = f"{np.mean(scores_2q):.2f} ± {np.std(scores_2q):.1f}" if scores_2q else 'N/A'
            d2 = f"{np.mean(drops_2q):.4f}" if drops_2q else 'N/A'
            s3 = f"{np.mean(scores_3q):.2f} ± {np.std(scores_3q):.1f}" if scores_3q else 'N/A'
            d3 = f"{np.mean(drops_3q):.4f}" if drops_3q else 'N/A'
            print(f"{env_name:<25} {st:<12} {level:<8.2f} {s2:<15} {d2:<10} {s3:<15} {d3:<10}")
    print()


In [ ]:
# Download all plots
# from google.colab import files
# for f in glob.glob('results/plot_*.png'):
#     files.download(f)

print('\nGenerated plots:')
for f in sorted(glob.glob('results/plot_*.png')):
    print(f'  {f}')